# EEG Biomarkers: An Example-Based Guide for Machine Learning

> **Purpose**: This notebook is a self-contained, visual, example-based walkthrough of the EEG biomarkers most important for neurodegeneration research. Each section teaches you what the biomarker measures biologically, what "normal" looks like, how it changes in disease, and how to compute and visualise it in Python.

---

## Why EEG Biomarkers Matter for ML

Raw EEG is ~500 Hz × 19 channels — far too high-dimensional for a model to learn clinical patterns from scratch on 88 subjects. Biomarkers compress the signal into a small set of **clinically-meaningful features** that:

- Are backed by decades of neuroscience literature
- Correspond to specific pathophysiological processes (neuronal loss, network disconnection, metabolic slowing)
- Allow your LLM/classifier to reason using the same vocabulary as neurologists

---

## Biomarkers Covered

| # | Biomarker | Domain | Clinical Significance |
|---|---|---|---|
| 1 | **Spectral Band Powers** | Frequency | Delta↑ Theta↑ Alpha↓ in AD |
| 2 | **Peak / Individual Alpha Frequency (IAF)** | Frequency | Slows from ~10 Hz → <8 Hz in AD |
| 3 | **Theta / Alpha Ratio** | Frequency | Hallmark spectral shift index |
| 4 | **Aperiodic Component (1/f slope)** | Frequency | Reflects E/I balance; frontier biomarker |
| 5 | **Permutation Entropy** | Complexity | Loss of temporal complexity in AD |
| 6 | **Sample Entropy** | Complexity | Regularity of the neural signal |
| 7 | **Spectral Entropy** | Complexity | How spread-out the power spectrum is |
| 8 | **Lempel-Ziv Complexity** | Complexity | Binary signal algorithmic complexity |
| 9 | **Posterior Alpha Coherence** | Connectivity | Network disconnection in AD |
| 10 | **Phase-Amplitude Coupling (PAC)** | Cross-frequency | Theta-gamma coupling, memory networks |

---

## Setup Note

This notebook uses the ds004504 dataset (88 subjects: AD, FTD, CN).  
Run the `eeg_hackathon.ipynb` first to confirm the data is present at `../data/ds004504/`.


## 0. Setup & Imports

In [ ]:
# Install any missing packages
import subprocess, sys
pkgs = ['mne', 'antropy', 'specparam', 'mne-connectivity', 'scipy', 'seaborn', 'tqdm']
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', p, '-q'], check=False)
print("Packages ready.")


In [ ]:
import warnings, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.signal import welch, coherence, butter, sosfiltfilt, hilbert
from scipy.integrate import simpson as simps
from tqdm.notebook import tqdm

import mne
mne.set_log_level('WARNING')

import antropy as ant
from specparam import SpectralModel, SpectralGroupModel

print(f"MNE:      {mne.__version__}")
import specparam; print(f"specparam: {specparam.__version__}")
print(f"antropy:  {ant.__version__}")

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
sns.set_style('whitegrid')
GROUP_COLORS = {'AD': '#E74C3C', 'FTD': '#F39C12', 'CN': '#2ECC71'}
BAND_COLORS  = {'delta':'#95A5A6','theta':'#F1C40F','alpha':'#27AE60',
                'beta':'#3498DB','gamma':'#9B59B6'}
BANDS        = {'delta':(0.5,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,45)}


In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
# Resolve dataset root — works whether you run from notebooks/ or repo root
candidates = [Path('data/ds004504'), Path('../data/ds004504')]
DATASET_ROOT = next((p.resolve() for p in candidates if p.exists()), candidates[1].resolve())
print(f"Dataset root: {DATASET_ROOT}")

# Participants metadata
df_participants = pd.read_csv(DATASET_ROOT / 'participants.tsv', sep='\t')
df_participants['Group'] = df_participants['Group'].map({'A':'AD','C':'CN','F':'FTD'})
s2g = dict(zip(df_participants['participant_id'], df_participants['Group']))
s2mmse = dict(zip(df_participants['participant_id'], df_participants['MMSE']))

def get_eeg_path(subject_id):
    return DATASET_ROOT / subject_id / 'eeg' / f'{subject_id}_task-eyesclosed_eeg.set'

available = [s for s in df_participants['participant_id'] if get_eeg_path(s).exists()]
print(f"Available subjects: {len(available)}/{len(df_participants)}")

def load_and_preprocess(subject_id, analysis_duration=120.0):
    """Load and preprocess one subject. Returns mne.io.Raw."""
    raw = mne.io.read_raw_eeglab(str(get_eeg_path(subject_id)), preload=True, verbose=False)
    raw.set_channel_types({ch: 'eeg' for ch in raw.ch_names})
    try:
        raw.set_montage(mne.channels.make_standard_montage('standard_1020'),
                        on_missing='ignore', verbose=False)
    except Exception:
        pass
    raw.filter(0.5, 45.0, method='fir', verbose=False)
    raw.set_eeg_reference('average', projection=False, verbose=False)
    t0 = min(60.0, raw.times[-1] - 30)
    raw.crop(tmin=t0, tmax=min(t0 + analysis_duration, raw.times[-1]))
    return raw

# ── Select one representative subject per group ───────────────────────────────
def pick_subject(group, n=0):
    """Pick the nth available subject from a group."""
    subs = [s for s in available if s2g.get(s) == group]
    return subs[n] if n < len(subs) else None

EXAMPLES = {g: pick_subject(g) for g in ['AD','CN','FTD']}
print("Example subjects:", EXAMPLES)

print("\nLoading example subjects...")
raws = {g: load_and_preprocess(s) for g, s in EXAMPLES.items() if s}
print("Done.")


---
## 1. What Does a Healthy Resting-State EEG Look Like?

Before studying what goes *wrong* in disease, we must build an intuition for what *healthy* EEG looks like.

### The Canonical Resting-State EEG (Eyes Closed)

A healthy adult with eyes closed produces:

| Frequency Band | Hz Range | Primary Generators | Eyes-Closed Pattern |
|---|---|---|---|
| **Delta** | 0.5–4 | Brainstem/thalamus | Minimal — present only during deep sleep |
| **Theta** | 4–8 | Hippocampus/frontal | Low amplitude; slightly elevated over frontal |
| **Alpha** | 8–13 | Thalamo-cortical loops | **Dominant** — large occipital/parietal waves |
| **Beta** | 13–30 | Cortex (motor/prefrontal) | Low-amplitude, diffuse |
| **Gamma** | 30–45 | Cortical circuits | Very low amplitude in scalp EEG |

### The Alpha Peak: Your Neural Heartbeat

The **Individual Alpha Frequency (IAF)** — the peak of the alpha power in resting EEG — is one of the most stable traits of the human brain:
- Healthy adults: **9–11 Hz** (mean ~10 Hz)
- Stable across decades of life in cognitively healthy individuals
- Genetically influenced (heritability ~0.8)
- Strong predictor of cognitive capacity and processing speed

> **Clinical red flag**: An alpha peak that *slows* over time, or is already below 8.5 Hz, is one of the earliest and most reliable EEG signs of neurodegeneration.


In [ ]:
# ── Side-by-side EEG and PSD for healthy (CN) subject ────────────────────────
cn_subj = EXAMPLES['CN']
raw_cn  = raws['CN']
sfreq   = raw_cn.info['sfreq']

fig = plt.figure(figsize=(18, 9))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

# — 1a: Raw EEG trace, 10 seconds ─────────────────────────────────────────────
ax_raw = fig.add_subplot(gs[0, :])
data, times = raw_cn[:, int(60*sfreq):int(70*sfreq)]
scale = 50e-6   # 50 µV per channel
cmap  = plt.cm.viridis(np.linspace(0, 1, len(raw_cn.ch_names)))
for i, (ch_data, color) in enumerate(zip(data, cmap)):
    offset = (len(raw_cn.ch_names) - 1 - i) * scale
    ax_raw.plot(times - times[0], ch_data + offset, color=color, linewidth=0.7, alpha=0.85)
ax_raw.set_yticks([(len(raw_cn.ch_names)-1-i)*scale for i in range(len(raw_cn.ch_names))])
ax_raw.set_yticklabels(raw_cn.ch_names, fontsize=7)
ax_raw.set_xlabel('Time (s)')
ax_raw.set_title(f'Healthy Control ({cn_subj}) — Raw EEG  |  Notice the prominent rhythmic alpha waves over posterior channels (O1, O2)',
                 fontweight='bold')

# — 1b: PSD with annotated bands ──────────────────────────────────────────────
ax_psd = fig.add_subplot(gs[1, 0])
avg_data = data.mean(axis=0)
f_cn, psd_cn = welch(avg_data, fs=sfreq, nperseg=int(sfreq*4))
mask = (f_cn >= 0.5) & (f_cn <= 45)
ax_psd.semilogy(f_cn[mask], psd_cn[mask], color=GROUP_COLORS['CN'], linewidth=2.5)
for bname, (flo, fhi) in BANDS.items():
    ax_psd.axvspan(flo, fhi, alpha=0.15, color=BAND_COLORS[bname], label=bname)
# Mark alpha peak
alpha_m = (f_cn >= 8) & (f_cn <= 13)
iaf_freq = f_cn[alpha_m][np.argmax(psd_cn[alpha_m])]
iaf_pow  = psd_cn[alpha_m].max()
ax_psd.annotate(f'IAF\n{iaf_freq:.1f} Hz', xy=(iaf_freq, iaf_pow),
                xytext=(iaf_freq+2, iaf_pow*3),
                arrowprops=dict(arrowstyle='->', color='black'),
                fontsize=9, fontweight='bold')
ax_psd.set_xlabel('Frequency (Hz)')
ax_psd.set_ylabel('PSD (log scale)')
ax_psd.set_title('Power Spectral Density\n(grand avg over channels)', fontweight='bold')
ax_psd.legend(fontsize=7, ncol=2)

# — 1c: Band power pie chart ──────────────────────────────────────────────────
ax_pie = fig.add_subplot(gs[1, 1])
band_powers_cn = {}
total_pow = 0
for bname, (flo, fhi) in BANDS.items():
    m = (f_cn >= flo) & (f_cn <= fhi)
    bp = simps(psd_cn[m], x=f_cn[m])
    band_powers_cn[bname] = bp
    total_pow += bp
fracs = [v/total_pow for v in band_powers_cn.values()]
wedge_props = dict(width=0.5)
wedges, texts, autotexts = ax_pie.pie(
    fracs, labels=[f'{b}\n({p*100:.1f}%)' for b,p in zip(band_powers_cn.keys(), fracs)],
    colors=[BAND_COLORS[b] for b in band_powers_cn],
    autopct='', startangle=90, wedgeprops=wedge_props
)
for txt in texts: txt.set_fontsize(8)
ax_pie.set_title('Band Power Distribution\n(Healthy Control, eyes closed)', fontweight='bold')

plt.suptitle('The Healthy Resting-State EEG — Your Baseline Reference', 
             fontsize=14, fontweight='bold')
plt.show()

print(f"\n✦ Healthy Control ({cn_subj}) alpha peak: {iaf_freq:.1f} Hz")
print(f"  Alpha dominates the spectrum, especially over posterior channels.")
print(f"  Delta and theta are minimal during alert rest.")


---
## 2. Biomarker 1 — Spectral Band Powers & Topographic Maps

### What it measures
Relative power in each classical frequency band, computed from the Welch power spectral density (PSD).  
**Relative** power (band ÷ total broadband) is used to normalise for recording gain and electrode impedance differences.

### The AD Spectral Signature
The Alzheimer's EEG shows a characteristic **"spectral shift to the left"**:

```
Healthy:   Delta↓   Theta↓   ALPHA↑↑   Beta—   Gamma—
AD:        DELTA↑   THETA↑   alpha↓    Beta↓   Gamma↓
```

This pattern reflects the gradual death of fast-oscillating cortical circuits and increasing
dominance of the brainstem/thalamic slow rhythms.

### Clinical Note
- **Delta increase**: reflects cortical neuronal loss and white matter degeneration
- **Theta increase**: reflects hippocampal and cholinergic pathway degeneration
- **Alpha decrease**: reflects thalamo-cortical loop dysfunction, reduced top-down modulation


In [ ]:
# ── Compute PSDs and band powers for all three groups ─────────────────────────

def get_psd_and_bands(raw):
    """Returns freqs, mean PSD across channels, and dict of relative band powers."""
    spectrum = raw.compute_psd(method='welch', fmin=0.5, fmax=45,
                               n_fft=2000, n_overlap=1000, verbose=False)
    psd, freqs = spectrum.get_data(return_freqs=True)  # (n_ch, n_freqs)
    mean_psd   = psd.mean(axis=0)
    
    broad_mask  = (freqs >= 0.5) & (freqs <= 45)
    total_power = simps(mean_psd[broad_mask], x=freqs[broad_mask])
    
    bp = {}
    for bname, (flo, fhi) in BANDS.items():
        m      = (freqs >= flo) & (freqs <= fhi)
        bp[bname] = simps(mean_psd[m], x=freqs[m]) / (total_power + 1e-15)
    
    return freqs, mean_psd, bp

group_psds  = {}
group_bands = {}
for group, raw in raws.items():
    f, psd, bands = get_psd_and_bands(raw)
    group_psds[group]  = (f, psd)
    group_bands[group] = bands


In [ ]:
# ── Figure 2: PSD comparison + band power bar chart ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# — Left: overlaid PSDs ────────────────────────────────────────────────────────
ax = axes[0]
for group in ['CN','AD','FTD']:
    f, psd = group_psds[group]
    mask   = (f >= 0.5) & (f <= 45)
    ax.semilogy(f[mask], psd[mask], color=GROUP_COLORS[group], linewidth=2.5, label=group)

# Shade frequency bands
for bname, (flo, fhi) in BANDS.items():
    ax.axvspan(flo, fhi, alpha=0.08, color=BAND_COLORS[bname])
    ax.text((flo+fhi)/2, ax.get_ylim()[0]*1.5 if ax.get_ylim()[0] > 0 else 1e-14,
            bname, ha='center', fontsize=7, color=BAND_COLORS[bname], fontweight='bold')

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power Spectral Density (log scale)')
ax.set_title('PSD: CN vs AD vs FTD\n(Grand mean across all channels)',
             fontweight='bold')
ax.legend(fontsize=10)

# Annotate the alpha slowdown
for group in ['CN','AD']:
    f_g, psd_g = group_psds[group]
    alpha_m = (f_g >= 7) & (f_g <= 13)
    iaf = f_g[alpha_m][np.argmax(psd_g[alpha_m])]
    ax.axvline(iaf, color=GROUP_COLORS[group], linestyle='--', alpha=0.6, linewidth=1.2,
               label=f'{group} IAF={iaf:.1f}Hz')

ax.legend(fontsize=8, ncol=2)

# — Right: relative band power grouped bars ────────────────────────────────────
ax2 = axes[1]
band_names  = list(BANDS.keys())
n_bands     = len(band_names)
n_groups    = 3
group_order = ['CN','AD','FTD']
bar_width   = 0.22
x           = np.arange(n_bands)

for gi, group in enumerate(group_order):
    vals = [group_bands[group][b] * 100 for b in band_names]
    bars = ax2.bar(x + gi*bar_width, vals, bar_width, label=group,
                   color=GROUP_COLORS[group], alpha=0.85, edgecolor='white')

ax2.set_xticks(x + bar_width)
ax2.set_xticklabels([b.capitalize() for b in band_names], fontsize=10)
ax2.set_ylabel('Relative Band Power (%)')
ax2.set_title('Relative Band Power by Group\n(% of total broadband 0.5–45 Hz)',
              fontweight='bold')
ax2.legend()

# Add difference annotations
for bi, band in enumerate(band_names):
    cn_v  = group_bands['CN'][band]  * 100
    ad_v  = group_bands['AD'][band]  * 100
    diff  = ad_v - cn_v
    color = '#C0392B' if diff > 0.5 else '#27AE60' if diff < -0.5 else 'gray'
    ax2.annotate(f'Δ{diff:+.1f}', xy=(bi + bar_width*0.5, max(cn_v, ad_v) + 0.3),
                 ha='center', fontsize=7, color=color, fontweight='bold')

plt.suptitle('Spectral Band Powers — The Alzheimer\'s Spectral Shift',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✦ Interpretation guide:")
print("  Delta (+) = cortical neuronal loss, white matter degeneration")
print("  Theta (+) = hippocampal/cholinergic pathway damage")
print("  Alpha (−) = thalamo-cortical loop dysfunction")
print("  AD shows the classic 'spectral shift left': power moves from fast → slow bands")


In [ ]:
# ── Figure 3: Topographic band power maps ────────────────────────────────────
# Shows WHERE on the scalp the band power changes are most prominent

fig, axes = plt.subplots(3, 5, figsize=(18, 9))
band_names  = list(BANDS.keys())
group_order = ['CN','AD','FTD']

# Compute per-channel band powers for each group
def channel_band_powers(raw):
    """Returns dict: band -> (n_channels,) relative power array."""
    spectrum   = raw.compute_psd(method='welch', fmin=0.5, fmax=45,
                                 n_fft=2000, n_overlap=1000, verbose=False)
    psd, freqs = spectrum.get_data(return_freqs=True)
    broad_mask  = (freqs >= 0.5) & (freqs <= 45)
    total_power = psd[:, broad_mask].mean(axis=1)
    bp = {}
    for bname, (flo, fhi) in BANDS.items():
        m = (freqs >= flo) & (freqs <= fhi)
        bp[bname] = psd[:, m].mean(axis=1) / (total_power + 1e-15)
    return bp, spectrum

ch_band_powers = {g: channel_band_powers(raws[g]) for g in group_order}

for ri, group in enumerate(group_order):
    bp, spectrum = ch_band_powers[group]
    for ci, band in enumerate(band_names):
        ax = axes[ri, ci]
        power_vals = bp[band]
        
        # Use MNE's plot_topomap
        mne.viz.plot_topomap(
            power_vals,
            raws[group].info,
            axes=ax,
            show=False,
            vlim=(power_vals.min(), power_vals.max()),
            cmap='RdYlGn_r' if band in ['delta','theta'] else 'RdYlGn',
            contours=4,
        )
        if ri == 0:
            ax.set_title(band.capitalize(), fontweight='bold', fontsize=10)
        if ci == 0:
            ax.set_ylabel(group, fontweight='bold', color=GROUP_COLORS[group], fontsize=11)
            ax.yaxis.label.set_visible(True)

plt.suptitle(
    'Topographic Band Power Maps\n'
    'Row = group, Column = frequency band  |  Warm=higher power, Cool=lower power',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

print("\n✦ Reading the topomaps:")
print("  CN: Large green occipital alpha (dominant eyes-closed rhythm)")
print("  AD: Reduced alpha topography; increased frontal/central theta")
print("  FTD: Stronger frontal involvement than AD")


---
## 3. Biomarker 2 — Individual Alpha Frequency (IAF) / Posterior Dominant Rhythm (PDR)

### What it is

The **Individual Alpha Frequency (IAF)** is the frequency of peak power in the alpha band (7–13 Hz).  
When measured over posterior electrodes (O1, O2, P3, P4, Pz) during eyes-closed rest, it is called the **Posterior Dominant Rhythm (PDR)**.

### Why it matters

The IAF reflects the natural oscillation frequency of **thalamocortical loops** — the bidirectional circuits between the thalamus and the cortex that regulate consciousness, attention, and memory encoding. These loops depend critically on:
- Intact myelination
- Functional cholinergic innervation (acetylcholine speeds these loops)
- Metabolic integrity of pyramidal neurons

In Alzheimer's disease, **cholinergic denervation** and **synaptic loss** reduce the speed of these loops → the dominant frequency slows.

### Normal reference ranges

| Population | IAF / PDR |
|---|---|
| Healthy adults 20–60 | 9.5–11.5 Hz |
| Healthy adults 60–80 | 8.5–10.5 Hz |
| MCI (prodromal AD) | 8.0–9.5 Hz |
| Mild-Moderate AD | < 8.0 Hz |

> **Key diagnostic threshold**: PDR < 8.0 Hz is used in many clinical guidelines as a flag for cortical dysfunction.  
> PDR < 7.5 Hz is considered severely abnormal.

### How it's measured

The simplest reliable estimator: take the **centroid** (or peak) of the alpha power spectrum over O1, O2, P3, P4, Pz, restricted to the 7–13 Hz range.


In [ ]:
# ── Compute IAF for a full cohort sample ─────────────────────────────────────
POSTERIOR_CHS = ['O1','O2','P3','P4','Pz']

def compute_iaf(raw, fmin=7.0, fmax=13.0):
    """
    Compute Individual Alpha Frequency via spectral peak over posterior channels.
    
    Returns
    -------
    iaf_peak    : float  peak frequency (Hz) — maximum of alpha PSD
    iaf_centroid: float  centre of gravity of alpha PSD (more robust)
    """
    sfreq = raw.info['sfreq']
    avail = [ch for ch in POSTERIOR_CHS if ch in raw.ch_names] or raw.ch_names[-4:]
    data  = raw.copy().pick(avail).get_data().mean(axis=0)
    
    freqs, psd = welch(data, fs=sfreq, nperseg=int(sfreq * 4))
    alpha_m    = (freqs >= fmin) & (freqs <= fmax)
    f_a, p_a   = freqs[alpha_m], psd[alpha_m]
    
    iaf_peak     = float(f_a[np.argmax(p_a)])
    iaf_centroid = float(np.sum(f_a * p_a) / (np.sum(p_a) + 1e-15))
    return iaf_peak, iaf_centroid


# Load a larger cohort sample (up to 10 per group) for a meaningful distribution
print("Computing IAF for a cohort sample...")
iaf_records = []
subject_counts = {'AD':0,'CN':0,'FTD':0}
max_per_group  = 12

for subj in tqdm(available, desc='IAF computation'):
    group = s2g.get(subj)
    if not group or subject_counts.get(group, 0) >= max_per_group:
        continue
    try:
        raw = load_and_preprocess(subj, analysis_duration=60)
        peak, centroid = compute_iaf(raw)
        iaf_records.append({'subject': subj, 'group': group,
                             'iaf_peak': peak, 'iaf_centroid': centroid,
                             'mmse': s2mmse.get(subj, np.nan)})
        subject_counts[group] = subject_counts.get(group,0) + 1
    except Exception as e:
        pass

iaf_df = pd.DataFrame(iaf_records)
print(f"\nComputed IAF for {len(iaf_df)} subjects")
print(iaf_df.groupby('group')[['iaf_peak','iaf_centroid']].mean().round(2))


In [ ]:
# ── Figure 4: IAF distribution and example spectra ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# — Left: IAF distribution (violin + strip) ───────────────────────────────────
ax = axes[0]
for gi, group in enumerate(['CN','AD','FTD']):
    data = iaf_df[iaf_df['group']==group]['iaf_peak']
    parts = ax.violinplot(data, positions=[gi], widths=0.6,
                          showmedians=True, showextrema=True)
    for pc in parts['bodies']:
        pc.set_facecolor(GROUP_COLORS[group])
        pc.set_alpha(0.6)
    parts['cmedians'].set_color(GROUP_COLORS[group])
    parts['cmedians'].set_linewidth(3)
    ax.scatter(np.full(len(data), gi) + np.random.uniform(-0.08, 0.08, len(data)),
               data, color=GROUP_COLORS[group], s=25, alpha=0.8, zorder=5)

ax.axhline(8.0, color='red', linestyle='--', alpha=0.6, linewidth=1.5,
           label='Clinical threshold (8 Hz)')
ax.axhline(9.5, color='gray', linestyle=':', alpha=0.5, linewidth=1.2,
           label='Normal lower bound (9.5 Hz)')
ax.set_xticks([0,1,2])
ax.set_xticklabels(['CN','AD','FTD'], fontsize=11)
ax.set_ylabel('Individual Alpha Frequency (Hz)')
ax.set_title('IAF Distribution by Group\n(Peak frequency, posterior channels)',
             fontweight='bold')
ax.legend(fontsize=8)
ax.set_ylim(5, 14)

# — Middle: posterior PSD zoom on alpha for each group ────────────────────────
ax2 = axes[1]
for group in ['CN','AD','FTD']:
    raw = raws[group]
    sfreq = raw.info['sfreq']
    avail = [ch for ch in POSTERIOR_CHS if ch in raw.ch_names]
    data  = raw.copy().pick(avail).get_data().mean(axis=0)
    f, psd = welch(data, fs=sfreq, nperseg=int(sfreq*4))
    mask   = (f >= 4) & (f <= 16)
    # Normalise each curve to its own max for visual comparison
    psd_n = psd[mask] / (psd[mask].max() + 1e-15)
    ax2.plot(f[mask], psd_n, color=GROUP_COLORS[group], linewidth=2.5, label=group)
    # Mark peak
    iaf_i = np.argmax(psd[mask])
    ax2.scatter(f[mask][iaf_i], psd_n[iaf_i],
                color=GROUP_COLORS[group], s=80, zorder=5, edgecolors='black')

ax2.axvline(8.0, color='red', linestyle='--', alpha=0.6, label='8 Hz threshold')
ax2.axvspan(8, 13, alpha=0.07, color=BAND_COLORS['alpha'], label='Alpha band')
ax2.set_xlabel('Frequency (Hz)')
ax2.set_ylabel('Normalised PSD')
ax2.set_title('Posterior Alpha Peak\n(Normalised to show peak shift)',
              fontweight='bold')
ax2.legend(fontsize=8)

# — Right: MMSE vs IAF scatter (cognitive correlate) ──────────────────────────
ax3 = axes[2]
for group in ['CN','AD','FTD']:
    sub = iaf_df[iaf_df['group']==group].dropna(subset=['mmse'])
    ax3.scatter(sub['mmse'], sub['iaf_peak'],
                color=GROUP_COLORS[group], s=60, alpha=0.8, label=group,
                edgecolors='white', linewidth=0.5)

valid = iaf_df.dropna(subset=['mmse','iaf_peak'])
r, p = stats.pearsonr(valid['mmse'], valid['iaf_peak'])
z = np.polyfit(valid['mmse'], valid['iaf_peak'], 1)
x_r = np.linspace(0, 30, 100)
ax3.plot(x_r, np.poly1d(z)(x_r), 'k--', alpha=0.4, linewidth=1.5,
         label=f'r={r:.2f}, p={p:.3f}')
ax3.axhline(8.0, color='red', linestyle='--', alpha=0.5)
ax3.set_xlabel('MMSE Score (cognitive severity)')
ax3.set_ylabel('IAF Peak (Hz)')
ax3.set_title('IAF vs Cognitive Severity\n(MMSE correlation)', fontweight='bold')
ax3.legend(fontsize=8)

plt.suptitle('Individual Alpha Frequency — The Signature of Cortical Slowing',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n✦ Pearson r(MMSE, IAF) = {r:.3f}  (p={p:.4f})")
print(f"   Positive correlation: higher MMSE (less impaired) → faster alpha")


---
## 4. Biomarker 3 — The Aperiodic Component (1/f Slope)

### What it is

When you plot EEG power on a log-log scale, the spectrum follows a roughly **straight descending line** — this is the **aperiodic** (1/f-like) component. Peaks above this line (like the alpha peak) are the **periodic** (oscillatory) components.

Until ~2018, most researchers simply examined oscillatory peaks. The **FOOOF/specparam** algorithm (Donoghue et al., 2020, *Nature Neuroscience*) revolutionised this by separating them:

```
log(PSD) = aperiodic_component + periodic_peaks
         = (offset − exponent × log(f)) + Σ Gaussians
```

### Why the 1/f exponent matters

The **aperiodic exponent** (slope steepness) is hypothesised to reflect the **excitation-inhibition (E/I) balance** of neural circuits:
- **Steeper slope** (larger exponent) → relatively more inhibition
- **Flatter slope** (smaller exponent) → relatively more excitation

**Frontier research (2023–2026)** links aperiodic changes in AD to:
- Tau pathology (flatter slope in preclinical AD)
- Synaptic dysfunction and E/I imbalance
- Distinct pattern from periodic (oscillatory) changes

> ⚠️ **Still debated**: Whether the aperiodic exponent is a *primary* AD biomarker or secondary to oscillatory changes. Worth including in your feature set regardless.

### Interpreting specparam output

```
Aperiodic params: [offset, exponent]
  offset    = log-power intercept (higher = more overall power)
  exponent  = slope steepness (typical healthy: 1.0–1.8)
  
Peaks: [center_frequency, power, bandwidth]
  Each Gaussian above the aperiodic baseline
```


In [ ]:
# ── Fit specparam model to each group's PSD ───────────────────────────────────

def fit_specparam(raw, freq_range=(1, 40)):
    """
    Fit the specparam (FOOOF) model to the mean PSD of a recording.
    Returns the fitted model and the aperiodic/peak parameters.
    """
    spectrum   = raw.compute_psd(method='welch', fmin=0.5, fmax=45,
                                 n_fft=2000, n_overlap=1000, verbose=False)
    psd, freqs = spectrum.get_data(return_freqs=True)
    mean_psd   = psd.mean(axis=0)
    
    fm = SpectralModel(
        peak_width_limits = [1.0, 8.0],
        max_n_peaks       = 6,
        min_peak_height   = 0.1,
        verbose           = False,
    )
    fm.fit(freqs, mean_psd, freq_range=freq_range)
    
    ap     = fm.get_params('aperiodic')       # [offset, exponent]
    peaks  = fm.get_params('peak', 'cf')      # peak center frequencies
    powers = fm.get_params('peak', 'pw')      # peak powers
    
    return fm, freqs, mean_psd, ap, peaks, powers


print("Fitting specparam models...")
sp_results = {}
for group, raw in raws.items():
    fm, freqs, mean_psd, ap, peaks, pows = fit_specparam(raw)
    sp_results[group] = {
        'fm': fm, 'freqs': freqs, 'mean_psd': mean_psd,
        'offset': ap[0], 'exponent': ap[1],
        'peaks': peaks if hasattr(peaks,'__len__') else [peaks],
        'peak_powers': pows if hasattr(pows,'__len__') else [pows],
    }
    print(f"  {group}: offset={ap[0]:.2f}  exponent={ap[1]:.3f}  peaks={np.round(peaks,1)}Hz")


In [ ]:
# ── Figure 5: Specparam decomposition side by side ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, group in zip(axes, ['CN','AD','FTD']):
    res   = sp_results[group]
    fm    = res['fm']
    freqs = res['freqs']
    color = GROUP_COLORS[group]

    # Raw PSD (log10)
    mask = (freqs >= 1) & (freqs <= 40)
    log_psd = np.log10(res['mean_psd'][mask] + 1e-30)
    ax.plot(freqs[mask], log_psd, color='gray', alpha=0.5, linewidth=1.5, label='Raw PSD')

    # Aperiodic fit (the straight line in log-log space)
    exponent = res['exponent']
    offset   = res['offset']
    ap_fit   = offset - exponent * np.log10(freqs[mask])
    ax.plot(freqs[mask], ap_fit, color=color, linewidth=2.5,
            linestyle='--', label=f'Aperiodic (exp={exponent:.2f})')

    # Full model fit
    # Regenerate model output
    fitted = fm.results.modeled_spectrum_
    f_fit  = fm.data.freqs_
    if f_fit is not None and fitted is not None:
        ax.plot(f_fit, fitted, color='black', linewidth=1.2, alpha=0.7, label='Full fit')

    # Mark peaks
    for cf in (res['peaks'] if hasattr(res['peaks'],'__iter__') else [res['peaks']]):
        if cf and not np.isnan(cf):
            ax.axvline(cf, color=color, linestyle=':', alpha=0.6, linewidth=1.2)
            ax.text(cf, ax.get_ylim()[1]*0.95 if ax.get_ylim()[1] != 0 else log_psd.max()*0.95,
                    f'{cf:.1f}', ha='center', fontsize=7, color=color, fontweight='bold')

    # Shade residuals (oscillatory peaks = area between raw and aperiodic)
    residuals = log_psd - ap_fit
    pos_mask  = residuals > 0
    ax.fill_between(freqs[mask], ap_fit, log_psd,
                    where=pos_mask, alpha=0.25, color=color,
                    label='Oscillatory peaks')

    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('log₁₀ PSD')
    ax.set_title(f'{group}\nAperiodic exp: {exponent:.3f}  |  Offset: {offset:.2f}',
                 color=color, fontweight='bold')
    ax.legend(fontsize=7)

plt.suptitle(
    'Specparam Decomposition — Separating Aperiodic (1/f) from Oscillatory Components\n'
    'Dashed line = 1/f fit  |  Shaded area = oscillatory peaks above aperiodic baseline',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

print("\n✦ How to read this plot:")
print("  Dashed line = 1/f aperiodic fit (reflects E/I balance)")
print("  Steeper slope = larger exponent = more inhibition-dominant")
print("  Shaded peaks = true oscillations (alpha, beta) above the noise floor")
print("  Specparam lets you measure the ALPHA PEAK without contamination from 1/f noise")


---
## 5. Biomarkers 4–7 — Signal Complexity & Entropy

### Why complexity matters in neurodegeneration

A healthy brain is a **complex, unpredictable, high-dimensional** dynamical system.
It constantly generates novel patterns — this is what allows it to respond flexibly to a
changing environment. As neurons die and networks simplify, the EEG becomes **more regular,
more repetitive, lower complexity**.

Think of it this way:
- A **random noise generator** has *maximum* complexity (every sample is independent)
- A **pure sine wave** has *minimum* complexity (perfectly predictable)
- A **healthy brain** sits somewhere between — structured but not repetitive
- A **diseased brain** shifts toward the sine-wave end — more regular, more predictable

This progression is captured by multiple **entropy** and **complexity** measures, each with slightly different sensitivities.

---

### The Four Measures Compared

| Measure | What it counts | Key properties | Good for |
|---|---|---|---|
| **Permutation Entropy** | Ordinal patterns (relative order of values) | Robust to noise, fast, parameter-light | General-purpose complexity |
| **Sample Entropy** | Probability of pattern recurrence | No self-matching bias, length-independent | Short/noisy signals |
| **Spectral Entropy** | How spread-out the power spectrum is | Frequency domain, fast | Power distribution complexity |
| **Lempel-Ziv Complexity** | Number of unique binary sub-patterns | Based on data compression | Temporal structure |

All four should decrease as AD/neurodegeneration progresses — but they capture different aspects
of complexity and are **partially independent**, making them valuable as a combined feature set.


In [ ]:
# ── Compute all four entropy/complexity measures ─────────────────────────────

def compute_all_entropy(raw):
    """
    Compute all four entropy/complexity biomarkers for a Raw recording.
    Returns dict of {measure: float}.
    """
    data  = raw.get_data()          # (n_ch, n_times)
    sfreq = raw.info['sfreq']

    # 1. Permutation Entropy (mean across channels)
    perm_vals = [ant.perm_entropy(ch, order=3, delay=1, normalize=True) for ch in data]

    # 2. Sample Entropy (mean across channels)
    #    r = 0.15 * std is the standard tolerance
    samp_vals = [ant.sample_entropy(ch, order=2) for ch in data]

    # 3. Spectral Entropy (Welch method via antropy)
    spec_vals = [ant.spectral_entropy(ch, sf=sfreq, method='welch',
                                      normalize=True) for ch in data]

    # 4. Lempel-Ziv Complexity (binarise at median)
    lzc_vals  = [ant.lziv_complexity((ch > np.median(ch)).astype(int),
                                     normalize=True) for ch in data]

    return {
        'perm_entropy':   float(np.mean(perm_vals)),
        'sample_entropy': float(np.nanmean(samp_vals)),
        'spectral_entropy': float(np.mean(spec_vals)),
        'lz_complexity':  float(np.mean(lzc_vals)),
    }


# Compute for example subjects
print("Computing entropy measures for example subjects...")
entropy_by_group = {}
for group, raw in raws.items():
    measures = compute_all_entropy(raw)
    entropy_by_group[group] = measures
    print(f"  {group}: perm={measures['perm_entropy']:.4f}  "
          f"sample={measures['sample_entropy']:.4f}  "
          f"spectral={measures['spectral_entropy']:.4f}  "
          f"lzc={measures['lz_complexity']:.4f}")


In [ ]:
# ── Figure 6: Entropy comparison — waveform examples + bar charts ────────────
fig = plt.figure(figsize=(18, 11))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.4)

# ── Top row: 3-second signal segments to visualise complexity visually ────────
groups = ['CN','AD','FTD']
T_SHOW = 3.0   # seconds to display

for gi, group in enumerate(groups):
    ax = fig.add_subplot(gs[0, gi])
    raw = raws[group]
    sfreq = raw.info['sfreq']
    # Use a middle channel (e.g., Cz or the ~10th channel)
    ch_idx = min(16, len(raw.ch_names)-1)  # Cz or similar
    n_samp = int(T_SHOW * sfreq)
    seg    = raw.get_data()[ch_idx, :n_samp]
    t      = np.arange(n_samp) / sfreq
    ax.plot(t, seg * 1e6, color=GROUP_COLORS[group], linewidth=1.0)  # convert to µV
    
    # Overlay a sliding-window complexity indicator as colour backdrop
    win = int(0.5 * sfreq)
    complexities = []
    for start in range(0, n_samp - win, win//2):
        chunk = seg[start:start+win]
        perm  = ant.perm_entropy(chunk, order=3, delay=1, normalize=True)
        complexities.append(perm)
    
    ent_measures = entropy_by_group[group]
    ax.set_title(
        f'{group}  ({raw.ch_names[ch_idx]})
'
        f'PermEnt={ent_measures["perm_entropy"]:.3f}  '
        f'LZC={ent_measures["lz_complexity"]:.3f}',
        color=GROUP_COLORS[group], fontweight='bold', fontsize=9
    )
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude (µV)')

# Add explanation panel
ax_exp = fig.add_subplot(gs[0, 3])
ax_exp.axis('off')
ax_exp.text(0.05, 0.9,
    'Complexity Intuition',
    fontsize=12, fontweight='bold', transform=ax_exp.transAxes
)
ax_exp.text(0.05, 0.1,
    'HIGH complexity:
'
    '  Irregular, hard to predict
'
    '  → Healthy, active brain

'
    'LOW complexity:
'
    '  Regular, repeating patterns
'
    '  → Neurodegeneration

'
    'All 4 entropy measures
'
    'decrease as AD progresses.
'
    'They capture different
'
    'aspects of signal structure.',
    fontsize=9, transform=ax_exp.transAxes,
    verticalalignment='bottom',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.4)
)

# ── Bottom row: bar charts for each measure ────────────────────────────────────
measures  = ['perm_entropy','sample_entropy','spectral_entropy','lz_complexity']
titles    = ['Permutation Entropy
(ordinal patterns)',
             'Sample Entropy
(pattern recurrence)',
             'Spectral Entropy
(power distribution)',
             'Lempel-Ziv Complexity
(binary patterns)']

for mi, (meas, title) in enumerate(zip(measures, titles)):
    ax = fig.add_subplot(gs[1, mi])
    vals   = [entropy_by_group[g][meas] for g in groups]
    colors = [GROUP_COLORS[g] for g in groups]
    bars   = ax.bar(groups, vals, color=colors, alpha=0.85, edgecolor='white', width=0.5)
    
    # Add value labels
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.3f}', ha='center', fontsize=8, fontweight='bold')
    
    # Highlight CN as reference
    ax.bar(['CN'], [entropy_by_group['CN'][meas]],
           color=GROUP_COLORS['CN'], alpha=0.85, edgecolor='white', width=0.5,
           linewidth=2)
    
    ax.set_title(title, fontweight='bold', fontsize=9)
    ax.set_ylabel('Value (normalised 0–1)' if mi == 0 else '')
    ax.set_ylim(0, max(vals) * 1.25)

plt.suptitle(
    'Signal Complexity & Entropy Measures\n'
    'All should be HIGHER in healthy controls than in AD/FTD',
    fontsize=13, fontweight='bold'
)
plt.show()


In [ ]:
# ── Figure 7: Per-channel entropy topomaps ───────────────────────────────────
# Shows WHERE complexity is reduced — typically worst in posterior regions in AD

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
measures  = ['perm_entropy','sample_entropy','spectral_entropy','lz_complexity']
col_titles = ['Permutation\nEntropy','Sample\nEntropy','Spectral\nEntropy','LZ\nComplexity']

print("Computing per-channel entropy for topomaps (takes ~1 min)...")
for ri, group in enumerate(groups):
    raw   = raws[group]
    data  = raw.get_data()
    sfreq = raw.info['sfreq']
    
    ch_entropy = {m: [] for m in measures}
    for ch_data in data:
        ch_entropy['perm_entropy'].append(
            ant.perm_entropy(ch_data, order=3, delay=1, normalize=True))
        ch_entropy['sample_entropy'].append(
            float(ant.sample_entropy(ch_data, order=2)))
        ch_entropy['spectral_entropy'].append(
            ant.spectral_entropy(ch_data, sf=sfreq, method='welch', normalize=True))
        ch_entropy['lz_complexity'].append(
            ant.lziv_complexity((ch_data > np.median(ch_data)).astype(int), normalize=True))
    
    for ci, meas in enumerate(measures):
        ax = axes[ri, ci]
        vals = np.array(ch_entropy[meas])
        # Handle NaNs
        vals = np.nan_to_num(vals, nan=np.nanmedian(vals))
        mne.viz.plot_topomap(
            vals, raw.info, axes=ax, show=False,
            vlim=(vals.min(), vals.max()),
            cmap='RdYlGn',   # green=high complexity, red=low
            contours=3,
        )
        if ri == 0:
            axes[0, ci].set_title(col_titles[ci], fontweight='bold', fontsize=9)
        if ci == 0:
            axes[ri, 0].set_ylabel(group, fontweight='bold',
                                   color=GROUP_COLORS[group], fontsize=11)

plt.suptitle(
    'Topographic Complexity Maps\n'
    'Green = high complexity (healthy), Red = low complexity (pathological)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

print("\n✦ Reading the complexity topomaps:")
print("  CN: Green broadly, especially occipital — high healthy complexity")
print("  AD: Red/yellow in posterior regions — loss of posterior complexity")
print("  FTD: More frontal involvement distinguishing from AD")


---
## 6. Biomarker 8 — Functional Connectivity (Coherence & PLI)

### What it measures

Functional connectivity captures the **statistical relationship between signals from different electrodes** — essentially asking: "Do these two brain regions oscillate together?"

The most commonly used measures are:

| Measure | What it captures | Volume conduction bias |
|---|---|---|
| **Coherence** | Linear correlation in frequency domain | High (inflated by volume conduction) |
| **Phase Lag Index (PLI)** | Phase consistency, ignoring zero-lag | Low (zero-lag = artefact) |
| **Imaginary Coherence** | Imaginary part of coherence only | Zero (immune by construction) |
| **wPLI** | Weighted PLI — more stable | Low |

For most EEG research with a clinical focus, **coherence** and **PLI** are the most reported.

### The AD Connectivity Signature

**Network disconnection** is one of the earliest and most consistent findings in AD:
- **Reduced alpha-band coherence** between frontal and posterior regions
- **Reduced inter-hemispheric coherence** (left-right)
- **Posterior disconnection**: O1-O2, P3-P4, and cross-temporal links are most affected
- Pattern reflects physical disruption of long-range white matter tracts

> This is thought to reflect **amyloid-β-induced synaptic dysfunction** in long-range projection neurons before local circuit failure becomes apparent.


In [ ]:
from mne_connectivity import spectral_connectivity_epochs

def compute_connectivity_matrix(raw, method='coh', fmin=8.0, fmax=13.0, epoch_len=4.0):
    """
    Compute full n×n connectivity matrix in a frequency band.
    Uses fixed-length epochs for reliable spectral estimation.
    
    Parameters
    ----------
    method  : 'coh' (coherence) or 'pli' (phase lag index)
    fmin, fmax : frequency band of interest (Hz)
    epoch_len  : epoch length in seconds
    
    Returns
    -------
    conn_mat : (n_ch, n_ch) ndarray
    ch_names : list
    """
    # Create fixed-length non-overlapping epochs
    events = mne.make_fixed_length_events(raw, duration=epoch_len)
    epochs = mne.Epochs(raw, events, tmin=0, tmax=epoch_len,
                        baseline=None, preload=True, verbose=False)
    
    conn = spectral_connectivity_epochs(
        epochs,
        method  = method,
        fmin    = fmin,
        fmax    = fmax,
        faverage= True,
        verbose = False,
    )
    data = conn.get_data(output='dense').squeeze()  # (n_ch, n_ch)
    # Symmetrise (upper triangle → both directions)
    data = (data + data.T) / 2
    np.fill_diagonal(data, 0)
    return data, raw.ch_names


print("Computing alpha-band connectivity matrices...")
conn_matrices = {}
for group, raw in raws.items():
    mat, chs = compute_connectivity_matrix(raw, method='coh', fmin=8, fmax=13)
    conn_matrices[group] = {'matrix': mat, 'ch_names': chs}
    mean_conn = mat[mat > 0].mean()
    print(f"  {group}: mean alpha coherence = {mean_conn:.4f}")


In [ ]:
# ── Figure 8: Connectivity matrices and comparison ───────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

groups_plot = ['CN','AD','FTD']
vmin_global = min(conn_matrices[g]['matrix'].min() for g in groups_plot)
vmax_global = max(conn_matrices[g]['matrix'].max() for g in groups_plot)

for ai, group in enumerate(groups_plot):
    ax    = axes[ai]
    mat   = conn_matrices[group]['matrix']
    chs   = conn_matrices[group]['ch_names']
    n_ch  = len(chs)
    color = GROUP_COLORS[group]
    
    im = ax.imshow(mat, cmap='RdYlGn', vmin=vmin_global, vmax=vmax_global,
                   aspect='auto', interpolation='nearest')
    ax.set_xticks(range(n_ch))
    ax.set_yticks(range(n_ch))
    ax.set_xticklabels(chs, rotation=90, fontsize=6)
    ax.set_yticklabels(chs, fontsize=6)
    plt.colorbar(im, ax=ax, label='Coherence', shrink=0.7)
    
    mean_val = mat[mat > 0].mean()
    ax.set_title(f'{group}\nMean coherence: {mean_val:.4f}',
                 color=color, fontweight='bold')

# — Difference plot: AD minus CN ───────────────────────────────────────────────
ax_diff = axes[3]
mat_cn  = conn_matrices['CN']['matrix']
mat_ad  = conn_matrices['AD']['matrix']
diff    = mat_ad - mat_cn
chs     = conn_matrices['CN']['ch_names']
n_ch    = len(chs)

vmax_d = np.abs(diff).max()
im_d   = ax_diff.imshow(diff, cmap='RdBu_r', vmin=-vmax_d, vmax=vmax_d, aspect='auto')
ax_diff.set_xticks(range(n_ch))
ax_diff.set_yticks(range(n_ch))
ax_diff.set_xticklabels(chs, rotation=90, fontsize=6)
ax_diff.set_yticklabels(chs, fontsize=6)
plt.colorbar(im_d, ax=ax_diff, label='Δ Coherence (AD−CN)', shrink=0.7)
ax_diff.set_title('AD − CN Difference\n(Blue=AD lower, Red=AD higher)',
                  fontweight='bold', color='#8E44AD')

plt.suptitle(
    'Alpha-Band (8–13 Hz) Functional Connectivity Matrices\n'
    'Green=high, Red=low coherence  |  AD shows systematic reduction in long-range connections',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

print("\n✦ Key observations:")
print("  CN: Strong coherence in alpha band, especially between adjacent electrodes")
print("  AD: Systematically lower coherence — network disconnection pattern")
print("  Difference map (AD−CN): Blue = connections specifically lost in AD")


In [ ]:
# ── Figure 9: Connectivity circle plot (most affected links) ─────────────────
# Visualise the most changed connections as a circular graph

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, group in zip(axes, ['CN','AD']):
    mat  = conn_matrices[group]['matrix'].copy()
    chs  = conn_matrices[group]['ch_names']
    n_ch = len(chs)
    color = GROUP_COLORS[group]
    
    # Circular layout
    angles = np.linspace(0, 2*np.pi, n_ch, endpoint=False)
    xs     = np.cos(angles)
    ys     = np.sin(angles)
    
    ax.set_aspect('equal')
    ax.axis('off')
    
    # Draw only top-quartile connections
    threshold = np.percentile(mat[mat > 0], 75)
    for i in range(n_ch):
        for j in range(i+1, n_ch):
            if mat[i, j] >= threshold:
                alpha_conn = (mat[i,j] - threshold) / (mat.max() - threshold + 1e-9)
                ax.plot([xs[i], xs[j]], [ys[i], ys[j]],
                        color=color, alpha=float(alpha_conn) * 0.6 + 0.1,
                        linewidth=float(mat[i,j]) * 3)
    
    # Draw channel labels
    for k, (ch, x, y) in enumerate(zip(chs, xs, ys)):
        ax.text(x * 1.18, y * 1.18, ch,
                ha='center', va='center', fontsize=7, fontweight='bold')
    
    # Draw nodes
    ax.scatter(xs, ys, s=40, c=color, zorder=5, edgecolors='white', linewidth=0.5)
    
    mean_val = mat[mat > 0].mean()
    ax.set_title(f'{group}\nAlpha coherence (top 25%)\nMean = {mean_val:.4f}',
                 color=color, fontweight='bold')

plt.suptitle(
    'Connectivity Circle Plots — Alpha Band (8–13 Hz)\n'
    'Each line = a strong connection; thicker/more opaque = stronger coherence',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()


---
## 7. Biomarker 9 — Phase-Amplitude Coupling (PAC)

### What it is

**Phase-Amplitude Coupling (PAC)** measures the relationship between:
- The **phase** of a slow oscillation (e.g., theta, 4–8 Hz)
- The **amplitude** (power) of a fast oscillation (e.g., gamma, 30–80 Hz)

When these are coupled, the amplitude of gamma fluctuates rhythmically with the theta phase — gamma bursts happen preferentially at specific theta phase angles.

### Why it matters in neurodegeneration

**Theta-gamma coupling** is the mechanistic basis of the **"neural code for working memory"** (Lisman & Idiart, 1995):
- **Theta rhythm** (4–8 Hz): organises the sequence of items in working memory
- **Gamma bursts** (30–80 Hz): represent individual items in working memory
- AD disrupts this hierarchy because **hippocampal theta generators are among the first structures to fail**

**Research findings (2023–2025)**:
- Reduced theta-gamma PAC in parahippocampal and temporal regions in AD
- PAC changes appear earlier than coherence changes in MCI
- MCI may show *paradoxical* PAC elevation early (compensatory hyperexcitability)

### How it's computed

The **Modulation Index (MI)** by Tort et al. (2010) is the most widely used PAC measure:
1. Extract phase of low-frequency signal (Hilbert transform)
2. Extract amplitude envelope of high-frequency signal (Hilbert transform)
3. Bin amplitudes by phase angle (18 bins of 20°)
4. Compute entropy of the amplitude distribution vs. uniform


In [ ]:
def compute_pac(data, sfreq, low_band=(4,8), high_band=(30,45), n_bins=18):
    """
    Compute Phase-Amplitude Coupling using the Modulation Index (Tort et al., 2010).
    
    Steps:
    1. Bandpass filter to extract low-frequency phase signal
    2. Bandpass filter to extract high-frequency amplitude signal
    3. Get instantaneous phase (Hilbert) of low, amplitude envelope of high
    4. Bin amplitude by phase angle
    5. Compute MI as KL-divergence from uniform distribution
    
    Parameters
    ----------
    data      : 1D array  time-series signal
    sfreq     : float     sampling frequency
    low_band  : (flo, fhi) for phase signal
    high_band : (flo, fhi) for amplitude signal
    n_bins    : int        number of phase bins
    
    Returns
    -------
    mi        : float  Modulation Index (0 = no coupling, >0 = coupling)
    amp_by_phase : (n_bins,) mean amplitude per phase bin
    """
    # 1. Bandpass filter for phase signal
    b_low = butter(4, low_band, btype='band', fs=sfreq, output='sos')
    sig_low = sosfiltfilt(b_low, data)
    
    # 2. Bandpass filter for amplitude signal  
    b_high = butter(4, high_band, btype='band', fs=sfreq, output='sos')
    sig_high = sosfiltfilt(b_high, data)
    
    # 3. Instantaneous phase and amplitude envelope
    phase = np.angle(hilbert(sig_low))
    amplitude = np.abs(hilbert(sig_high))
    
    # 4. Bin amplitudes by phase
    phase_bins = np.linspace(-np.pi, np.pi, n_bins + 1)
    amp_by_phase = np.zeros(n_bins)
    for k in range(n_bins):
        in_bin = (phase >= phase_bins[k]) & (phase < phase_bins[k+1])
        amp_by_phase[k] = amplitude[in_bin].mean() if in_bin.sum() > 0 else 0
    
    # 5. Modulation Index (KL-divergence from uniform)
    # Normalise to probability distribution
    p = amp_by_phase / (amp_by_phase.sum() + 1e-15)
    q = np.ones(n_bins) / n_bins  # uniform
    # KL divergence
    kl_div = np.sum(p * np.log((p + 1e-15) / (q + 1e-15)))
    mi = kl_div / np.log(n_bins)  # Normalised 0–1
    
    return float(mi), amp_by_phase


# Compute PAC for each group (one central/temporal channel)
print("Computing PAC for example subjects...")
pac_results = {}
for group, raw in raws.items():
    sfreq = raw.info['sfreq']
    # Use a temporal channel (T3/T4 if present, else fall back)
    temporal_chs = ['T3','T4','T5','T6','C3','C4','Cz']
    ch_use = next((c for c in temporal_chs if c in raw.ch_names), raw.ch_names[8])
    ch_idx = raw.ch_names.index(ch_use)
    data   = raw.get_data()[ch_idx]
    
    # Theta-gamma PAC
    mi_tg, amp_tg = compute_pac(data, sfreq, low_band=(4,8), high_band=(30,45))
    # Delta-alpha PAC
    mi_da, amp_da = compute_pac(data, sfreq, low_band=(1,4), high_band=(8,13))
    
    pac_results[group] = {
        'ch': ch_use,
        'theta_gamma_MI': mi_tg,
        'theta_gamma_amp': amp_tg,
        'delta_alpha_MI': mi_da,
        'delta_alpha_amp': amp_da,
    }
    print(f"  {group} ({ch_use}):  theta-gamma MI = {mi_tg:.5f}  "
          f"delta-alpha MI = {mi_da:.5f}")


In [ ]:
# ── Figure 10: PAC phase-amplitude comodulograms ─────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# ── Top row: Amplitude-by-phase polar plots (theta-gamma) ─────────────────────
n_bins    = 18
bin_edges = np.linspace(-np.pi, np.pi, n_bins + 1)
bin_ctrs  = (bin_edges[:-1] + bin_edges[1:]) / 2

for ci, group in enumerate(['CN','AD','FTD']):
    ax = axes[0, ci]
    amp = pac_results[group]['theta_gamma_amp']
    amp_n = amp / (amp.max() + 1e-15)  # normalise to max
    
    # Bar plot (phase on x-axis)
    ax.bar(bin_ctrs, amp_n, width=2*np.pi/n_bins,
           color=GROUP_COLORS[group], alpha=0.8, edgecolor='white')
    
    # Uniform reference line
    ax.axhline(1/n_bins, color='gray', linestyle='--', linewidth=1.2, alpha=0.7,
               label='Uniform (no PAC)')
    
    mi = pac_results[group]['theta_gamma_MI']
    ax.set_xlabel('Theta Phase (radians)')
    ax.set_ylabel('Normalised Gamma Amplitude')
    ax.set_title(f'{group} — Theta-Gamma PAC\nModulation Index = {mi:.5f}',
                 color=GROUP_COLORS[group], fontweight='bold')
    ax.set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax.set_xticklabels(['-π', '-π/2', '0', 'π/2', 'π'], fontsize=8)
    ax.legend(fontsize=7)

# ── Bottom row: Full comodulogram (phase × amplitude frequency sweep) ─────────
def compute_comodulogram(data, sfreq,
                          phase_bands=None, amp_bands=None):
    """Compute PAC for a grid of phase×amplitude frequency pairs."""
    if phase_bands is None:
        phase_bands = [(f, f+2) for f in range(2, 12, 2)]
    if amp_bands is None:
        amp_bands   = [(f, f+10) for f in range(10, 80, 10)]
    
    n_phase = len(phase_bands)
    n_amp   = len(amp_bands)
    comod   = np.zeros((n_phase, n_amp))
    
    for pi, pb in enumerate(phase_bands):
        for ai, ab in enumerate(amp_bands):
            if ab[0] > sfreq/2 - 5:
                continue
            try:
                mi, _ = compute_pac(data, sfreq, low_band=pb, high_band=ab)
                comod[pi, ai] = mi
            except Exception:
                comod[pi, ai] = np.nan
    return comod, phase_bands, amp_bands

phase_bands = [(f, f+2) for f in range(2, 14, 2)]
amp_bands   = [(f, f+10) for f in range(10, 50, 5)]

for ci, group in enumerate(['CN','AD','FTD']):
    ax = axes[1, ci]
    raw = raws[group]; sfreq = raw.info['sfreq']
    ch_idx = raw.ch_names.index(pac_results[group]['ch'])
    data   = raw.get_data()[ch_idx]
    
    comod, pb, ab = compute_comodulogram(data, sfreq, phase_bands, amp_bands)
    
    im = ax.imshow(comod, aspect='auto', origin='lower',
                   cmap='hot_r', interpolation='bilinear',
                   vmin=0, vmax=comod[~np.isnan(comod)].max())
    plt.colorbar(im, ax=ax, label='MI', shrink=0.8)
    
    phase_labels = [f'{p[0]}-{p[1]}' for p in pb]
    amp_labels   = [f'{a[0]}-{a[1]}' for a in ab]
    ax.set_yticks(range(len(pb)));  ax.set_yticklabels(phase_labels, fontsize=7)
    ax.set_xticks(range(len(ab)));  ax.set_xticklabels(amp_labels, rotation=45, fontsize=7)
    ax.set_xlabel('Amplitude Frequency (Hz)')
    ax.set_ylabel('Phase Frequency (Hz)')
    ax.set_title(f'{group} Comodulogram\n(hottest = strongest PAC)',
                 color=GROUP_COLORS[group], fontweight='bold')

plt.suptitle(
    'Phase-Amplitude Coupling (PAC) — Theta-Gamma Cross-Frequency Coupling\n'
    'Top: amplitude distribution by theta phase  |  Bottom: full frequency-pair grid',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

print("\n✦ Interpreting PAC:")
print("  Modulation Index (MI) > 0 = gamma amplitude is NOT uniformly distributed across theta phase")
print("  Peaks in the phase plot = gamma bursts prefer a specific theta phase angle")
print("  Comodulogram hot spot = which phase × amplitude pair is most coupled")
print("  AD should show REDUCED theta-gamma MI vs CN (hippocampal circuit failure)")


---
## 8. Putting It All Together — The Biomarker Profile

Now that we understand each biomarker individually, the real power comes from combining them into a **multi-dimensional profile** and learning to read the signature patterns.

### The AD Pattern (Summary)

```
Biomarker               AD vs CN          Biological meaning
────────────────────────────────────────────────────────────────
Delta power             ↑ Increased       Cortical degeneration
Theta power             ↑ Increased       Hippocampal/cholinergic failure
Alpha power             ↓ Decreased       Thalamo-cortical dysfunction
IAF / PDR               ↓ Slowed (<8 Hz)  Cholinergic denervation
Aperiodic exponent      ↕ Variable        E/I imbalance
Permutation entropy     ↓ Decreased       Loss of temporal complexity
Sample entropy          ↓ Decreased       Increased signal regularity
Spectral entropy        ↓ Decreased       Narrowed spectral distribution
LZ complexity           ↓ Decreased       More regular/compressible signal
Alpha coherence         ↓ Decreased       Long-range network disconnection
Theta-gamma PAC         ↓ Decreased       Working memory circuit failure
```

### AD vs FTD Differentiation

FTD primarily affects frontal and temporal lobes first, rather than parietal/posterior (as in AD):
- FTD: **Frontal delta/theta** disproportionately elevated vs posterior
- AD: More symmetric posterior slowing and connectivity loss
- Both: Reduced overall complexity, but different spatial distribution


In [ ]:
# ── Figure 11: Radar/spider chart — full biomarker profile per group ──────────

def normalize_0_1(val, min_val, max_val):
    return (val - min_val) / (max_val - min_val + 1e-15)

# Gather all biomarker values for each group
biomarker_values = {}
for group, raw in raws.items():
    f, mean_psd, bp = get_psd_and_bands(raw)
    entropy          = compute_all_entropy(raw)
    conn_mat, _      = compute_connectivity_matrix(raw, fmin=8, fmax=13)
    
    # PDR
    avail = [c for c in POSTERIOR_CHS if c in raw.ch_names]
    post  = raw.copy().pick(avail).get_data().mean(axis=0)
    f_p, psd_p = welch(post, fs=raw.info['sfreq'], nperseg=int(raw.info['sfreq']*4))
    pdr_mask = (f_p >= 4) & (f_p <= 14)
    pdr = float(f_p[pdr_mask][np.argmax(psd_p[pdr_mask])])
    
    # Specparam exponent
    fm, *_ = fit_specparam(raw)
    ap = fm.get_params('aperiodic')
    
    biomarker_values[group] = {
        'Delta power':      bp['delta'],
        'Theta power':      bp['theta'],
        'Alpha power':      bp['alpha'],
        'IAF/PDR (Hz)':     pdr,
        '1/f Exponent':     ap[1],
        'Perm Entropy':     entropy['perm_entropy'],
        'Sample Entropy':   entropy['sample_entropy'],
        'Spectral Entropy': entropy['spectral_entropy'],
        'LZ Complexity':    entropy['lz_complexity'],
        'Alpha Coherence':  float(conn_mat[conn_mat > 0].mean()),
    }

print("Collected all biomarker values:")
for group, bm in biomarker_values.items():
    print(f"  {group}:")
    for k, v in bm.items():
        print(f"    {k:25s}: {v:.4f}")


In [ ]:
# ── Figure 12: Spider/Radar chart ────────────────────────────────────────────
biomarker_names = list(biomarker_values['CN'].keys())
N = len(biomarker_names)

# For the radar, we need to normalise each feature to [0,1]
# For Delta/Theta power, higher = worse (AD-like), so we invert for display
invert_features = {'Delta power', 'Theta power'}

# Get global min/max per feature across groups
feat_min = {f: min(biomarker_values[g][f] for g in ['CN','AD','FTD'])
            for f in biomarker_names}
feat_max = {f: max(biomarker_values[g][f] for g in ['CN','AD','FTD'])
            for f in biomarker_names}

def get_radar_val(group, feat):
    v = normalize_0_1(biomarker_values[group][feat], feat_min[feat], feat_max[feat])
    if feat in invert_features:
        v = 1 - v  # invert so that "healthy = outer"
    return v

# Compute radar values
angles  = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for group in ['CN','AD','FTD']:
    values  = [get_radar_val(group, f) for f in biomarker_names]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2.5, label=group,
            color=GROUP_COLORS[group])
    ax.fill(angles, values, alpha=0.10, color=GROUP_COLORS[group])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(biomarker_names, fontsize=8, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75])
ax.set_yticklabels(['25%','50%','75%'], fontsize=7, color='gray')
ax.grid(color='gray', alpha=0.3)
ax.set_title(
    'EEG Biomarker Radar Profile\n'
    'OUTER = healthy direction  |  AD/FTD should be smaller',
    fontweight='bold', fontsize=11, pad=25
)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)

plt.tight_layout()
plt.show()

print("\n✦ How to read the radar:")
print("  All features scaled so that OUTER EDGE = healthier/normal direction")
print("  CN (green) should be largest — most complex, best connected, fastest alpha")
print("  AD (red) typically contracts inward — reduced complexity and connectivity")


In [ ]:
# ── Figure 13: Biomarker interpretation guide ─────────────────────────────────
# A final summary panel — the "cheat sheet" for reading biomarker reports

fig, ax = plt.subplots(figsize=(14, 9))
ax.axis('off')

# Table data
headers = ['Biomarker', 'CN (Healthy)', 'AD', 'FTD', 'Clinical Meaning']
rows = [
    ['IAF / PDR',         '9–11 Hz',     '< 8 Hz',    '8–9 Hz (mild)',
     'Thalamo-cortical loop speed; slows with cholinergic loss'],
    ['Delta power',       'Very low',     'Elevated',  'Elevated (frontal)',
     'Cortical degeneration, white matter damage'],
    ['Theta power',       'Low',          'Elevated',  'Elevated',
     'Hippocampal/cholinergic circuit failure'],
    ['Alpha power',       'Dominant',     'Reduced',   'Preserved longer',
     'Top-down thalamic modulation; first to go in AD'],
    ['Theta/Alpha ratio', '< 0.5',        '> 1.0',     'Variable',
     'Summary index of spectral shift'],
    ['1/f Exponent',      '1.0–1.8',     'Debated',   'Debated',
     'E/I balance; steeper=more inhibition'],
    ['Perm Entropy',      '0.85–0.95',   '0.65–0.80', '0.70–0.85',
     'Ordinal temporal complexity; ↓ = more regular'],
    ['Sample Entropy',    'Higher',       'Lower',     'Lower',
     'Pattern recurrence; ↓ = more predictable'],
    ['Spectral Entropy',  'Higher',       'Lower',     'Lower',
     'PSD flatness; ↓ = dominated by fewer frequencies'],
    ['LZ Complexity',     '0.15–0.25',   '0.08–0.15', '0.10–0.18',
     'Binary signal richness; ↓ = more compressible'],
    ['Alpha Coherence',   'Higher',       'Lower',     'Lower (frontal)',
     'Long-range synchrony; network disconnection in AD'],
    ['Theta-γ PAC (MI)',  'Moderate',    'Reduced',   'Variable',
     'Working memory circuits; hippocampal theta drives γ'],
]

col_widths  = [0.14, 0.12, 0.11, 0.16, 0.47]
col_x       = [0.0, 0.14, 0.26, 0.37, 0.53]
row_height  = 0.065
header_y    = 0.95
header_colors = ['#2C3E50','#27AE60','#E74C3C','#F39C12','#7F8C8D']
text_colors   = ['white','white','white','white','white']

# Draw header
for col_i, (h, cx, cw, hc, tc) in enumerate(
        zip(headers, col_x, col_widths, header_colors, text_colors)):
    ax.add_patch(plt.Rectangle((cx, header_y - 0.04), cw - 0.005, 0.055,
                               transform=ax.transAxes, fc=hc, ec='white', lw=1))
    ax.text(cx + cw/2, header_y - 0.012, h,
            ha='center', va='center', fontsize=9, fontweight='bold',
            color=tc, transform=ax.transAxes)

# Draw rows
for ri, row in enumerate(rows):
    y_pos = header_y - 0.04 - (ri+1) * row_height
    bg    = '#F9F9F9' if ri % 2 == 0 else '#FDFEFE'
    for ci, (cell, cx, cw) in enumerate(zip(row, col_x, col_widths)):
        ax.add_patch(plt.Rectangle((cx, y_pos), cw - 0.005, row_height - 0.005,
                                   transform=ax.transAxes, fc=bg, ec='#EAECEE', lw=0.5))
        fc = '#000000'
        if ci == 2:   fc = GROUP_COLORS['AD']
        elif ci == 3: fc = GROUP_COLORS['FTD']
        elif ci == 1: fc = GROUP_COLORS['CN']
        ax.text(cx + 0.005, y_pos + row_height/2, cell,
                ha='left', va='center', fontsize=7.5, color=fc,
                transform=ax.transAxes, wrap=True)

ax.set_title(
    'EEG Biomarker Interpretation Cheat Sheet\n'
    'For Clinical Pattern Recognition in Neurodegeneration',
    fontsize=14, fontweight='bold', pad=10
)
plt.tight_layout()
plt.show()


---
## 9. What Biomarkers to Feed Your ML Model

### Feature engineering recommendations

Having learned all these biomarkers, here is what the literature supports as the **best feature set** for a classifier trained on the ds004504 dataset:

#### Core features (consistently significant across studies)
1. `iaf_hz` — Individual Alpha Frequency (Hz)
2. `delta_power` — Relative delta power
3. `theta_power` — Relative theta power
4. `alpha_power` — Relative alpha power
5. `theta_alpha_ratio` — Spectral shift index
6. `posterior_coherence_alpha` — Mean pairwise alpha coherence over O1,O2,P3,P4,Pz
7. `lz_complexity` — LZ complexity (mean across channels)

#### Supplementary features (add robustness)
8. `perm_entropy` — Permutation entropy
9. `spectral_entropy` — Spectral entropy
10. `aperiodic_exponent` — 1/f slope (specparam)
11. `beta_power` — Relative beta power (FTD differentiator)
12. `theta_gamma_PAC` — Theta-gamma modulation index (temporal channels)

#### Spatial features (if you have enough data)
- Per-region band powers (frontal, temporal, parietal, occipital)
- Asymmetry indices (left vs right hemisphere)
- Region-specific coherence (frontal-posterior vs temporal-occipital)

### For the hackathon LLM pipeline specifically

Your biomarker summaries should include:
- **Binary flags** for each feature (above/below clinical threshold)
- **Z-scores** relative to the CN group norms
- **Spatial annotations** (frontal vs posterior involvement)

This gives the LLM enough structure to reason about *where* the pathology is, not just *whether* it exists.


In [ ]:
# ── Generate a complete, LLM-ready biomarker summary for one subject ──────────

def full_biomarker_summary(raw, subject_id, group_label, mmse, cn_norms=None):
    """
    Compute the full biomarker suite and generate an interpretable text report.
    This is the complete feature extraction pipeline used in Stage 1.
    """
    sfreq = raw.info['sfreq']
    
    # 1. Band powers
    _, _, bp = get_psd_and_bands(raw)
    
    # 2. IAF/PDR
    avail = [c for c in POSTERIOR_CHS if c in raw.ch_names]
    post  = raw.copy().pick(avail).get_data().mean(axis=0)
    f_p, psd_p = welch(post, fs=sfreq, nperseg=int(sfreq*4))
    pdr_m = (f_p >= 4) & (f_p <= 14)
    pdr   = float(f_p[pdr_m][np.argmax(psd_p[pdr_m])])
    
    # 3. Specparam exponent
    fm, *_ = fit_specparam(raw)
    ap_exp = float(fm.get_params('aperiodic')[1])
    
    # 4. Entropy
    ent = compute_all_entropy(raw)
    
    # 5. Connectivity
    conn_mat, _ = compute_connectivity_matrix(raw, fmin=8, fmax=13)
    mean_coh    = float(conn_mat[conn_mat > 0].mean())
    
    # 6. PAC (temporal channel)
    temporal_chs = ['T3','T4','C3','C4']
    ch_pac = next((c for c in temporal_chs if c in raw.ch_names), raw.ch_names[8])
    data_pac = raw.get_data()[raw.ch_names.index(ch_pac)]
    pac_mi, _ = compute_pac(data_pac, sfreq)
    
    # Assemble summary
    features = {
        'iaf_pdr_hz':           pdr,
        'delta_power_pct':      bp['delta'] * 100,
        'theta_power_pct':      bp['theta'] * 100,
        'alpha_power_pct':      bp['alpha'] * 100,
        'beta_power_pct':       bp['beta']  * 100,
        'gamma_power_pct':      bp['gamma'] * 100,
        'theta_alpha_ratio':    bp['theta'] / (bp['alpha'] + 1e-15),
        'aperiodic_exponent':   ap_exp,
        'perm_entropy':         ent['perm_entropy'],
        'sample_entropy':       ent['sample_entropy'],
        'spectral_entropy':     ent['spectral_entropy'],
        'lz_complexity':        ent['lz_complexity'],
        'alpha_coherence':      mean_coh,
        'theta_gamma_pac_mi':   pac_mi,
    }
    
    # Binary flags
    flags = {
        'PDR_slowed':                features['iaf_pdr_hz'] < 8.0,
        'delta_elevated':            features['delta_power_pct'] > 15,
        'theta_elevated':            features['theta_power_pct'] > 20,
        'alpha_reduced':             features['alpha_power_pct'] < 20,
        'theta_alpha_ratio_high':    features['theta_alpha_ratio'] > 1.0,
        'complexity_reduced':        features['lz_complexity'] < 0.15,
        'entropy_reduced':           features['perm_entropy'] < 0.80,
        'connectivity_reduced':      features['alpha_coherence'] < 0.25,
        'pac_reduced':               features['theta_gamma_pac_mi'] < 0.005,
    }
    
    # Text summary
    report = f"""
Subject: {subject_id}  |  True Group: {group_label}  |  MMSE: {mmse}

═══ QUANTITATIVE BIOMARKERS ═══════════════════════════════════════
Posterior Dominant Rhythm (PDR):   {features['iaf_pdr_hz']:.2f} Hz
  [Normal: 9-11 Hz  |  Slowed: <8 Hz  |  Severely slowed: <7.5 Hz]
  
Band Powers (relative, % of 0.5-45 Hz total):
  Delta  (0.5-4 Hz):  {features['delta_power_pct']:.1f}%   [Normal: <10%]
  Theta  (4-8 Hz):    {features['theta_power_pct']:.1f}%   [Normal: <15%]
  Alpha  (8-13 Hz):   {features['alpha_power_pct']:.1f}%   [Normal: >25%]
  Beta   (13-30 Hz):  {features['beta_power_pct']:.1f}%
  Gamma  (30-45 Hz):  {features['gamma_power_pct']:.1f}%
  Theta/Alpha ratio:  {features['theta_alpha_ratio']:.3f}  [Normal: <0.5, AD: >1.0]

Aperiodic (1/f) exponent:          {features['aperiodic_exponent']:.3f}
  [Reflects E/I balance; healthy range ~1.0-1.8]

Signal Complexity (all normalised 0-1; higher=more complex/healthy):
  Permutation Entropy:   {features['perm_entropy']:.4f}   [Normal: 0.85-0.95]
  Sample Entropy:        {features['sample_entropy']:.4f}   [Normal: higher values]
  Spectral Entropy:      {features['spectral_entropy']:.4f}   [Normal: 0.3-0.6]
  Lempel-Ziv Complexity: {features['lz_complexity']:.4f}   [Normal: 0.15-0.25]

Functional Connectivity:
  Alpha coherence (mean): {features['alpha_coherence']:.4f}   [Normal: >0.3]
  
Cross-Frequency Coupling:
  Theta-Gamma PAC (MI):   {features['theta_gamma_pac_mi']:.5f}   [Normal: >0.005]

═══ BINARY FLAGS ═══════════════════════════════════════════════════"""
    for flag, val in flags.items():
        status = '⚠ ABNORMAL' if val else '✓ normal'
        report += f"\n  {flag:35s}: {status}"
    
    n_flags = sum(flags.values())
    report += f"\n\nTotal abnormal flags: {n_flags}/{len(flags)}"
    if n_flags >= 5:
        report += "  →  HIGH suspicion for neurodegeneration"
    elif n_flags >= 3:
        report += "  →  MODERATE suspicion; additional clinical evaluation recommended"
    else:
        report += "  →  LOW suspicion based on EEG biomarkers"
    
    return features, flags, report


# Run for each example group
for group, raw in raws.items():
    subj  = EXAMPLES[group]
    mmse  = s2mmse.get(subj, '?')
    feats, flags, report = full_biomarker_summary(raw, subj, group, mmse)
    print(report)
    print()


---
## Summary

### What you learned in this notebook

1. **Normal EEG**: The healthy eyes-closed resting EEG is dominated by alpha (9–11 Hz) over occipital regions, with minimal slow-wave activity
2. **Band powers**: AD shows a systematic left-spectral-shift — delta/theta rise, alpha falls
3. **IAF/PDR**: The most clinically reliable single biomarker; slows from ~10 Hz to <8 Hz in AD, driven by cholinergic denervation
4. **Specparam/FOOOF**: Separates the 1/f "noise floor" from true oscillations — lets you measure the alpha peak without 1/f contamination; aperiodic exponent reflects E/I balance
5. **Entropy measures**: All four (permutation, sample, spectral, LZ) decrease with neurodegeneration — they capture different but complementary aspects of temporal and spectral complexity
6. **Connectivity**: AD produces widespread alpha-band network disconnection, especially posterior long-range connections
7. **PAC**: Theta-gamma coupling (working memory circuits) is reduced in AD; the comodulogram lets you survey all frequency-pair combinations at once

### Key takeaways for your ML pipeline

- Use **relative** band powers (not absolute) to normalise across recordings
- Always compute **IAF from posterior channels only** — frontal contamination distorts it
- **Specparam** should be preferred over raw PSD peaks — it accounts for the 1/f background
- Combine **both frequency domain and complexity** features — they are partially independent
- Include **spatial information** (frontal vs posterior) when differentiating AD from FTD
- For the LLM stage, **binary flags + z-scores** are more useful than raw values

### Recommended further reading

- Donoghue et al. (2020) *Parameterizing neural power spectra into periodic and aperiodic components* — Nature Neuroscience
- Dauwels et al. (2010) *Diagnosis of Alzheimer's disease from EEG signals: where are we standing?* — Current Alzheimer Research  
- Miltiadous et al. (2023) *DICE-net: A Novel Convolution-Transformer Architecture for Alzheimer Detection in EEG Signals* — IEEE Access (this dataset's paper)
- Klimesch (1999) *EEG alpha and theta oscillations reflect cognitive and memory performance: a review and analysis* — Brain Research Reviews
- Stam et al. (2009) *Graph theoretical analysis of magnetoencephalographic functional connectivity in Alzheimer's disease* — Brain
